In [13]:
# =========================================================
# CELL 1: CÀI ĐẶT THƯ VIỆN & XÁC THỰC GCP
# =========================================================
!pip install pyspark gcsfs -q

from google.colab import auth
auth.authenticate_user()
print("Xác thực GCP thành công!")

Xác thực GCP thành công!


In [14]:
# =============================================================
# CELL 2 — KHỞI TẠO SPARKSESSION
# =============================================================
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window
import gcsfs
import gc
import time
import os
import signal
import subprocess

# Dừng session cũ nếu còn tồn tại
try:
    if 'spark' in vars() and spark is not None:
        spark.stop()
        print(' Đã dừng SparkSession cũ.')
except BaseException:
    pass

def _build_spark() -> SparkSession:
    return (
        SparkSession.builder
        .appName('AmazonReviews_Final_Pipeline')
        .config('spark.jars.packages', 'com.google.cloud.bigdataoss:gcs-connector:hadoop3-2.2.5,com.google.cloud.spark:spark-bigquery-with-dependencies_2.12:0.34.0')
        .config('spark.hadoop.fs.gs.impl', 'com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem')
        .config('spark.driver.memory', '8g')           # Giảm để tránh sập Colab
        .config('spark.executor.memory', '8g')
        .config('spark.sql.shuffle.partitions', '200') # Tăng để chia nhỏ dữ liệu
        .config('spark.sql.autoBroadcastJoinThreshold', '250MB')
        .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer') # Nén dữ liệu tốt hơn
        .getOrCreate()
    )

spark = _build_spark()
spark.sparkContext.setLogLevel('ERROR')
print(f'SparkSession khởi tạo thành công! (version {spark.version})')

 Đã dừng SparkSession cũ.
SparkSession khởi tạo thành công! (version 4.0.2)


In [15]:
# =============================================================
# CELL 3 — HÀM RESTART SPARK KHI JVM CRASH
# =============================================================

def _force_kill_java() -> None:
    """Tìm và kill tất cả process 'java' đang chạy trong Colab VM."""
    try:
        result = subprocess.run(
            ['pgrep', '-f', 'pyspark'],
            capture_output=True, text=True
        )
        pids = [int(p) for p in result.stdout.strip().split() if p.isdigit()]
        for pid in pids:
            try:
                os.kill(pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
        if pids:
            print(f' Đã kill {len(pids)} JVM process: {pids}')
    except BaseException:
        pass


def restart_spark(wait: int = 12) -> None:
    """
    Dừng SparkSession hiện tại, force-kill JVM còn sót, chờ `wait` giây,
    rồi khởi động lại session mới. Cập nhật biến toàn cục `spark`.

    FIX v4:
      - Dùng `except BaseException` thay vì `except Exception` để bắt
        Py4JNetworkError (kế thừa BaseException trực tiếp)
      - Gọi _force_kill_java() để đảm bảo cổng JVM được giải phóng
        trước khi _build_spark() chạy
    """
    global spark
    print(f'Đang restart SparkSession (chờ {wait}s)...')

    try:
        spark.stop()
    except BaseException:
        pass

    _force_kill_java()

    time.sleep(wait)
    gc.collect()

    spark = _build_spark()
    spark.sparkContext.setLogLevel('ERROR')
    print('SparkSession đã được khởi động lại!')


def is_jvm_crash(exc: BaseException) -> bool:
    """
    Trả True nếu exception là do JVM/py4j crash.
    Kiểm tra cả tên class lẫn message vì Py4J có nhiều subclass.
    """
    crash_types = {
        'ConnectionRefusedError',
        'BrokenPipeError',
        'Py4JNetworkError',
        'Py4JJavaError',
        'Py4JError',
    }
    crash_msgs = [
        'connection refused',
        'answer from java side is empty',
        'java gateway process exited',
        'broken pipe',
        'an error occurred while calling',
    ]
    type_name = type(exc).__name__
    msg       = str(exc).lower()

    return (
        type_name in crash_types
        or any(s in msg for s in crash_msgs)
    )


def spark_safe_clear_cache() -> None:
    """Gọi spark.catalog.clearCache() an toàn — bỏ qua nếu JVM đã chết."""
    try:
        spark.catalog.clearCache()
    except BaseException:   # <-- FIX: bảo vệ finally block
        pass


def spark_safe_unpersist(df) -> None:
    """Gọi df.unpersist() an toàn — bỏ qua nếu JVM đã chết."""
    try:
        if df is not None:
            df.unpersist()
    except BaseException:
        pass


print('restart_spark(), is_jvm_crash(), spark_safe_clear_cache() đã sẵn sàng.')

restart_spark(), is_jvm_crash(), spark_safe_clear_cache() đã sẵn sàng.


In [16]:
# =============================================================
# CELL 4 — CẤU HÌNH ĐƯỜNG DẪN GCS & QUÉT DANH MỤC
# =============================================================
fs = gcsfs.GCSFileSystem()

BUCKET         = 'amazon-reviews-lakehouse-data'
bronze_base    = f'{BUCKET}/bronze-zone'
silver_base    = f'{BUCKET}/silver-zone'
NUM_PARTITIONS = 20    # Chia nhỏ để mỗi task dùng ít RAM hơn


def scan_folders(gcs_path: str) -> list:
    """Quét và trả về danh sách tên thư mục hợp lệ trên GCS."""
    try:
        entries = fs.ls(gcs_path)
    except FileNotFoundError:
        print(f'Không tìm thấy: {gcs_path}')
        return []
    folders = [e.split('/')[-1] for e in entries if fs.isdir(e)]
    return [
        f for f in folders
        if f and not f.startswith('_') and not f.startswith('.')
    ]


meta_folders   = scan_folders(f'{bronze_base}/meta-data/')
review_folders = scan_folders(f'{bronze_base}/review-data/')

print(f'Meta   : {len(meta_folders):>3} danh mục — {meta_folders}')
print(f'Review : {len(review_folders):>3} danh mục — {review_folders}')

Meta   :  31 danh mục — ['All_Beauty', 'Amazon_Fashion', 'Appliances', 'Arts_Crafts_and_Sewing', 'Automotive', 'Books', 'CDs_and_Vinyl', 'Cell_Phones_and_Accessories', 'Clothing_Shoes_and_Jewelry', 'Digital_Music', 'Electronics', 'Gift_Cards', 'Grocery_and_Gourmet_Food', 'Health_and_Household', 'Health_and_Personal_Care', 'Home_and_Kitchen', 'Industrial_and_Scientific', 'Kindle_Store', 'Magazine_Subscriptions', 'Movies_and_TV', 'Musical_Instruments', 'Office_Products', 'Patio_Lawn_and_Garden', 'Pet_Supplies', 'Software', 'Sports_and_Outdoors', 'Subscription_Boxes', 'Tools_and_Home_Improvement', 'Toys_and_Games', 'Unknown', 'Video_Games']
Review :  31 danh mục — ['All_Beauty', 'Amazon_Fashion', 'Appliances', 'Arts_Crafts_and_Sewing', 'Automotive', 'Books', 'CDs_and_Vinyl', 'Cell_Phones_and_Accessories', 'Clothing_Shoes_and_Jewelry', 'Digital_Music', 'Electronics', 'Gift_Cards', 'Grocery_and_Gourmet_Food', 'Health_and_Household', 'Health_and_Personal_Care', 'Home_and_Kitchen', 'Industria

In [17]:
# =============================================================
# CELL 7 — [FIXED] KIỂM TRA DUNG LƯỢNG & SẮP XẾP DANH MỤC
# =============================================================

def get_folder_size_mb(path):
    """Tính dung lượng chính xác bằng cách quét tất cả file bên trong."""
    try:
        # Đảm bảo đường dẫn có gs:// cho gcsfs
        if not path.startswith('gs://'):
            path = f"gs://{path}"

        # Quét chi tiết các file
        files = fs.ls(path, detail=True)
        if not files: return 0

        total_bytes = sum(f['size'] for f in files if f['type'] == 'file')
        return total_bytes / (1024 * 1024)
    except:
        return 0

def rank_categories_by_size():
    # Lấy danh mục chung
    common_cats = list(set(meta_folders).intersection(set(review_folders)))
    results = []

    print("📊 Đang quét dung lượng thực tế trên GCS...")
    for cat in common_cats:
        s_rev = get_folder_size_mb(f"{silver_base}/review-data/{cat}")
        s_meta = get_folder_size_mb(f"{silver_base}/meta-data/{cat}")
        total = s_rev + s_meta

        # Chỉ thêm nếu thực sự có dữ liệu (tránh folder rác 0MB)
        if total > 0:
            results.append({'category': cat, 'total_mb': total})
        else:
            # Nếu vẫn ra 0, in ra để debug
            print(f"⚠️ Cảnh báo: {cat} không tìm thấy dữ liệu hoặc 0MB")

    # Sắp xếp: Nhẹ nhất lên đầu
    ranked_list = sorted(results, key=lambda x: x['total_mb'])

    print(f"\n🏆 DANH SÁCH {len(ranked_list)} DANH MỤC ƯU TIÊN (NHẸ -> NẶNG):")
    for i, item in enumerate(ranked_list, 1):
        print(f"{i:<2}. {item['category']:<30} | {item['total_mb']:>10.2f} MB")

    return [item['category'] for item in ranked_list]

# Cập nhật danh sách ưu tiên
sorted_categories = rank_categories_by_size()

📊 Đang quét dung lượng thực tế trên GCS...

🏆 DANH SÁCH 31 DANH MỤC ƯU TIÊN (NHẸ -> NẶNG):
1 . Subscription_Boxes             |       2.12 MB
2 . Magazine_Subscriptions         |       8.07 MB
3 . Gift_Cards                     |       9.68 MB
4 . Digital_Music                  |      29.10 MB
5 . Health_and_Personal_Care       |      59.54 MB
6 . All_Beauty                     |      82.35 MB
7 . Appliances                     |     228.96 MB
8 . Amazon_Fashion                 |     291.39 MB
9 . Musical_Instruments            |     431.76 MB
10. Software                       |     446.60 MB
11. Industrial_and_Scientific      |     645.92 MB
12. Video_Games                    |     708.20 MB
13. CDs_and_Vinyl                  |    1054.80 MB
14. Arts_Crafts_and_Sewing         |    1067.07 MB
15. Grocery_and_Gourmet_Food       |    1429.69 MB
16. Office_Products                |    1497.44 MB
17. Toys_and_Games                 |    1861.40 MB
18. Pet_Supplies                   |    20

In [18]:
# # =============================================================
# # CELL 5 — XEM CHI TIẾT DỮ LIỆU VÙNG BẠC (SILVER INSPECTION)
# # =============================================================

# def peek_silver_data(category_name: str, num_rows: int = 5):
#     """
#     Đọc và hiển thị cấu trúc, dữ liệu mẫu của một danh mục tại vùng Bạc.
#     """
#     print(f"\n" + "="*60)
#     print(f"🔍 ĐANG KIỂM TRA DANH MỤC: {category_name}")
#     print("="*60)

#     # Đường dẫn dựa trên config trong file Untitled3.ipynb của bạn
#     rev_path  = f"gs://{silver_base}/review-data/{category_name}/*"
#     meta_path = f"gs://{silver_base}/meta-data/{category_name}/*"

#     try:
#         # 1. Đọc dữ liệu
#         df_rev  = spark.read.parquet(rev_path)
#         df_meta = spark.read.parquet(meta_path)

#         # 2. Hiển thị Schema (Cấu trúc dữ liệu)
#         print("\n📜 [REVIEW DATA SCHEMA]")
#         df_rev.printSchema()

#         print("\n📜 [METADATA SCHEMA]")
#         df_meta.printSchema()

#         # 3. Thống kê nhanh số lượng
#         print("\n📊 [STATISTICS]")
#         print(f" - Tổng số Reviews : {df_rev.count():,}")
#         print(f" - Tổng số Sản phẩm: {df_meta.count():,}")

#         # 4. Hiển thị mẫu (Dùng toPandas để bảng hiện đẹp trên Colab)
#         print(f"\n👀 {num_rows} DÒNG REVIEW MẪU:")
#         display(df_rev.limit(num_rows).toPandas())

#         print(f"\n👀 {num_rows} DÒNG METADATA MẪU:")
#         display(df_meta.limit(num_rows).toPandas())

#     except Exception as e:
#         if is_jvm_crash(e):
#             print("❌ JVM crashed! Đang restart Spark...")
#             restart_spark()
#         else:
#             print(f"❌ Lỗi khi đọc dữ liệu: {e}")

# # Chạy thử với danh mục đầu tiên tìm thấy
# if meta_folders:
#     peek_silver_data(meta_folders[0])

In [19]:
# =============================================================
# CELL 6 — [OPTIMIZED] THỰC HIỆN JOIN THEO THỨ TỰ ƯU TIÊN
# =============================================================
from pyspark.sql.functions import broadcast

# Đường dẫn folder joined
silver_joined_path = f'gs://{silver_base}/silver-joined'

def process_and_save_joined_data():
    # 1. Kiểm tra các danh mục ĐÃ xử lý xong trên GCS để bỏ qua
    try:
        processed_cats = scan_folders(silver_joined_path)
    except:
        processed_cats = []

    # 2. Lọc ra những danh mục chưa làm, theo thứ tự từ nhẹ đến nặng
    todo_categories = [c for c in sorted_categories if c not in processed_cats]

    print(f"🚀 Tổng: {len(sorted_categories)} | Đã xong: {len(processed_cats)} | Còn lại: {len(todo_categories)}")
    print(f"📍 Bắt đầu Join từ danh mục nhẹ nhất còn lại...")

    for cat in todo_categories:
        print(f"\n🔹 Processing [{cat}]...")
        start_time = time.time()

        try:
            # Đọc Silver với select tối giản
            df_rev = spark.read.parquet(f"gs://{silver_base}/review-data/{cat}/*") \
                .select("user_id", "parent_asin", "rating", "timestamp", "verified_purchase")

            df_meta = spark.read.parquet(f"gs://{silver_base}/meta-data/{cat}/*") \
                .select("parent_asin", "price", "main_category")

            # Thực hiện Broadcast Join (Vì đã xếp từ nhẹ, meta sẽ rất nhỏ)
            df_enriched = df_rev.join(broadcast(df_meta), on="parent_asin", how="inner")

            # Lưu vào silver-joined
            target_path = f"{silver_joined_path}/{cat}"
            df_enriched.write.mode("overwrite").parquet(target_path)

            duration = time.time() - start_time
            print(f" ✅ Saved {cat} ({duration:.1f}s)")

            # GIẢI PHÓNG RAM TRIỆT ĐỂ
            spark_safe_unpersist(df_rev)
            spark_safe_unpersist(df_meta)
            spark_safe_unpersist(df_enriched)
            spark_safe_clear_cache()
            gc.collect()

        except BaseException as e:
            if is_jvm_crash(e):
                print(f"⚠️ JVM Crash tại {cat}. Đang tự động restart Spark...")
                restart_spark()
                # Sau khi restart, chạy lại hàm này để tiếp tục danh mục đang dở
                return process_and_save_joined_data()
            else:
                print(f"❌ Lỗi tại {cat}: {e}")
                continue

# Thực thi
if sorted_categories:
    process_and_save_joined_data()
else:
    print("❌ Không có danh mục nào để xử lý. Vui lòng kiểm tra lại Cell 7.")

Không tìm thấy: gs://amazon-reviews-lakehouse-data/silver-zone/silver-joined
🚀 Tổng: 31 | Đã xong: 0 | Còn lại: 31
📍 Bắt đầu Join từ danh mục nhẹ nhất còn lại...

🔹 Processing [Subscription_Boxes]...
 ✅ Saved Subscription_Boxes (13.8s)

🔹 Processing [Magazine_Subscriptions]...
 ✅ Saved Magazine_Subscriptions (5.1s)

🔹 Processing [Gift_Cards]...
 ✅ Saved Gift_Cards (12.4s)

🔹 Processing [Digital_Music]...
 ✅ Saved Digital_Music (6.1s)

🔹 Processing [Health_and_Personal_Care]...
 ✅ Saved Health_and_Personal_Care (6.0s)

🔹 Processing [All_Beauty]...
 ✅ Saved All_Beauty (9.4s)

🔹 Processing [Appliances]...
 ✅ Saved Appliances (15.3s)

🔹 Processing [Amazon_Fashion]...
 ✅ Saved Amazon_Fashion (19.8s)

🔹 Processing [Musical_Instruments]...
 ✅ Saved Musical_Instruments (21.4s)

🔹 Processing [Software]...
 ✅ Saved Software (22.7s)

🔹 Processing [Industrial_and_Scientific]...
 ✅ Saved Industrial_and_Scientific (29.0s)

🔹 Processing [Video_Games]...
 ✅ Saved Video_Games (29.0s)

🔹 Processing [CDs

In [21]:
# =============================================================
# CELL 8 — UNION MASTER TABLE (GZIP) + TỐI ƯU TỐC ĐỘ & LOGGING
# =============================================================
import time

def create_master_table_optimized():
    silver_joined_path = f'gs://{silver_base}/silver-joined'
    master_table_path  = f'gs://{silver_base}/silver-master-table'

    print(f"🚀 BẮT ĐẦU TIẾN TRÌNH UNION MASTER TABLE")
    print(f"📂 Nguồn: {silver_joined_path}")
    print(f"📦 Đích  : {master_table_path}")
    print("-" * 50)

    overall_start = time.time()

    try:
        # 1. ĐỌC DỮ LIỆU (Lazy Loading)
        # Spark chưa thực sự đọc dữ liệu ở bước này, chỉ quét metadata.
        print(f"[{time.strftime('%H:%M:%S')}] 1. Đang quét metadata từ GCS...")
        df_master = spark.read.parquet(f"{silver_joined_path}/*")

        # 2. TỐI ƯU PHÂN VÙNG (REPARTITION)
        # 574M dòng cần khoảng 400-500 partitions để mỗi task xử lý ~1.5 triệu dòng.
        # Điều này giúp tận dụng tối đa CPU của máy ảo và tránh quá tải RAM.
        num_partitions = 400
        print(f"[{time.strftime('%H:%M:%S')}] 2. Đang phân phối lại dữ liệu (Repartition to {num_partitions})...")
        df_master = df_master.repartition(num_partitions)

        # 3. GHI DỮ LIỆU VỚI LOGGING TIẾN ĐỘ
        # Lưu ý: Gzip tốn CPU để nén nên sẽ lâu hơn Snappy.
        print(f"[{time.strftime('%H:%M:%S')}] 3. Đang thực hiện ghi Master Table (GZIP)...")
        print("   (Quá trình này có thể mất 15-30 phút tùy tốc độ GCS)")

        write_start = time.time()
        df_master.write.mode("overwrite") \
            .option("compression", "gzip") \
            .parquet(master_table_path)
        write_duration = time.time() - write_start

        # 4. KIỂM TRA VÀ TỔNG KẾT
        print(f"[{time.strftime('%H:%M:%S')}] 4. Đang đếm tổng số dòng (Final Count)...")
        total_rows = df_master.count()

        overall_duration = time.time() - overall_start

        print("\n" + "="*50)
        print("✅ HOÀN TẤT PIPELINE UNION")
        print(f"⏱️ Tổng thời gian: {overall_duration/60:.2f} phút")
        print(f"⚡ Thời gian ghi Gzip: {write_duration/60:.2f} phút")
        print(f"📊 Tổng số bản ghi: {total_rows:,}")

        # Lấy dung lượng file để log
        try:
            master_size = get_folder_size_mb(master_table_path)
            print(f"💾 Dung lượng Master (GZIP): {master_size:.2f} MB")
        except: pass
        print("="*50)

        # Dọn dẹp bộ nhớ
        spark_safe_unpersist(df_master)
        spark_safe_clear_cache()
        gc.collect()

    except BaseException as e:
        if is_jvm_crash(e):
            print("\n❌ JVM CRASH! Đang tự động restart...")
            restart_spark()
            # Gợi ý: Nếu crash ở đây, bạn nên tăng số lượng partitions lên 600
        else:
            print(f"\n❌ LỖI NGHIÊM TRỌNG: {str(e)}")

# Thực thi
create_master_table_optimized()

🚀 BẮT ĐẦU TIẾN TRÌNH UNION MASTER TABLE
📂 Nguồn: gs://amazon-reviews-lakehouse-data/silver-zone/silver-joined
📦 Đích  : gs://amazon-reviews-lakehouse-data/silver-zone/silver-master-table
--------------------------------------------------
[04:23:59] 1. Đang quét metadata từ GCS...
[04:25:04] 2. Đang phân phối lại dữ liệu (Repartition to 400)...
[04:25:04] 3. Đang thực hiện ghi Master Table (GZIP)...
   (Quá trình này có thể mất 15-30 phút tùy tốc độ GCS)
[05:22:55] 4. Đang đếm tổng số dòng (Final Count)...

✅ HOÀN TẤT PIPELINE UNION
⏱️ Tổng thời gian: 63.45 phút
⚡ Thời gian ghi Gzip: 57.85 phút
📊 Tổng số bản ghi: 473,459,775
💾 Dung lượng Master (GZIP): 15864.01 MB
